In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim

In [3]:
spark = get_sparkSession("Load silver.fact_inventory")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "fact_inventory"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_fact_inventory.ipynb"
_bronze_table = "bronze.fact_inventory"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.fact_inventory


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze.raw_customer and de-duplicating the data

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

df_dedup = df.withColumn("_rn", expr("row_number() over (partition by inventory_date, product_id, store, channel sort by created_on desc)")) \
             .where("_rn = 1") \
             .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Bronze Table Data count : 6
SPARK-APP: Table count after de-dupe : 6


In [10]:
df_dedup.show(10)
df_dedup.printSchema()


+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|inventory_date|product_id| store|channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|          created_on|          created_by|         modified_on|         modified_by|       source_file|
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|      20250101|     P1001|AMZ_US|    MKT|     USD|             100|               10|                90|       350|2026-01-29 00:29:...|raw_fact_inventor...|2026-01-29 00:29:...|raw_fact_inventor...|fact_inventory.csv|
|      20250101|     P1002|  EBAY|    MKT|     USD|             100|               10|                90|       350|2026

In [11]:
#Formating the data

df_ld = df_dedup.withColumn("inventory_date", to_date("inventory_date", 'yyyyMMdd')) \
                .withColumn("cost_price", expr("CAST(cost_price as decimal(18,2))")) \
                .withColumn("quantity_on_hand", expr("CAST(quantity_on_hand as int)")) \
                .withColumn("quantity_reserved", expr("CAST(quantity_reserved as int)")) \
                .withColumn("quantity_available", expr("CAST(quantity_available as int)"))


In [12]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("product_id", upper(trim("product_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel"))) \
             .withColumn("currency", upper(trim("currency")))


In [13]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [14]:
df_ld.show(10)
df_ld.printSchema()

+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|inventory_date|product_id| store|channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|          created_on|          created_by|         modified_on|         modified_by|       source_file|
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|    2025-01-01|     P1001|AMZ_US|    MKT|     USD|             100|               10|                90|    350.00|2026-01-29 01:05:...|silver_fact_inven...|2026-01-29 01:05:...|silver_fact_inven...|fact_inventory.csv|
|    2025-01-01|     P1002|  EBAY|    MKT|     USD|             100|               10|                90|    350.00|2026

In [15]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.fact_inventory and load_controller


In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|inventory_date|product_id| store|channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|          created_on|          created_by|         modified_on|         modified_by|       source_file|
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|    2025-01-01|     P1001|AMZ_US|    MKT|     USD|             100|               10|                90|    350.00|2026-01-29 01:05:...|silver_fact_inven...|2026-01-29 01:05:...|silver_fact_inven...|fact_inventory.csv|
|    2025-01-01|     P1002|  EBAY|    MKT|     USD|             100|               10|                90|    350.00|2026

In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+--------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|    table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+--------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|fact_inventory|full-load| overwrite|2026-01-29 01:05:...|    success|           6|2026-01-29 01:05:...|silver_fact_inven...|2026-01-29 01:05:...|silver_fact_inven...|
+------+-----------+--------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |6            |
+-------+-------------+



In [19]:
spark.stop()